# Comparación de modelos — #22
Se leen automáticamente todos los `docs/results/*_comparison.csv`: cada modelo aporta sus filas con las columnas del contrato común. Los archivos de trials no se incluyen. La columna `source_csv` identifica el origen de cada fila; referencias Dummy repetidas se conservan explícitamente, sin deduplicar configuraciones de forma silenciosa.

La selección se basa en la media de F1 macro de validación, no en el F1 agregado OOF. Los informes OOF describen train y no son nested CV. Se exige el mismo snapshot, hash de datos y configuración de CV, además de scores válidos y esquema común; si hay archivos incompatibles, se corrigen antes de comparar.

La tabla no incluye resultados del test reservado. Al final se elige el champion aplicando la regla acordada (mejor F1 macro medio de validación entre los modelos con gap < 0,05). Solo ese champion se evalúa en test, una vez, en `08b_champion_test.ipynb`. La baseline tiene exposición histórica de esa partición; el equipo debe considerarla al interpretar una evaluación final.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
RESULT_COLUMNS = [
    'model', 'cv_train_f1_macro', 'cv_validation_f1_macro', 'cv_validation_std',
    'gap', 'gap_under_005', 'cv_fit_seconds', 'evaluation_set', 'oof_accuracy',
    'oof_f1_macro', 'oof_extreme_error_rate', 'oof_seconds', 'search_seconds',
    'params', 'cv', 'snapshot', 'code_commit', 'raw_sha256',
]
paths = sorted((ROOT / 'docs/results').glob('*_comparison.csv'))
if not paths:
    raise FileNotFoundError('No hay archivos docs/results/*_comparison.csv para comparar')
frames = []
expected_columns = None
for path in paths:
    frame = pd.read_csv(path)
    held_out_columns = [c for c in frame.columns if c.startswith('test_')]
    if held_out_columns:
        raise ValueError(f'{path.name}: columnas de test reservadas no permitidas: {held_out_columns}')
    missing = set(RESULT_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name}: faltan columnas comunes: {sorted(missing)}')
    if 'source_csv' in frame.columns:
        raise ValueError(f'{path.name}: source_csv es una columna reservada para el origen')
    if expected_columns is None:
        expected_columns = set(frame.columns)
    elif set(frame.columns) != expected_columns:
        raise ValueError(f'{path.name}: las columnas deben coincidir con el resto de CSV de comparación')
    if frame.empty or frame['model'].isna().any() or frame['model'].astype(str).str.strip().eq('').any():
        raise ValueError(f'{path.name}: se necesita al menos una fila con nombre de modelo')
    if not frame['evaluation_set'].eq('train_oof').all():
        raise ValueError(f'{path.name}: los diagnósticos deben proceder de train_oof')
    for column in ['cv_train_f1_macro', 'cv_validation_f1_macro', 'cv_validation_std',
                   'oof_accuracy', 'oof_f1_macro', 'oof_extreme_error_rate']:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
        if not frame[column].between(0, 1).all():
            raise ValueError(f'{path.name}: {column} debe ser finito, no nulo y estar entre 0 y 1')
    frame['gap'] = pd.to_numeric(frame['gap'], errors='raise')
    if not np.isfinite(frame['gap']).all() or not np.allclose(
        frame['gap'], frame['cv_train_f1_macro'] - frame['cv_validation_f1_macro'], atol=1e-6,
    ):
        raise ValueError(f'{path.name}: gap no coincide con media train menos media validación')
    gap_flag = frame['gap_under_005'].map({True: True, False: False, 'True': True, 'False': False})
    if gap_flag.isna().any() or not gap_flag.eq(frame['gap'].lt(0.05)).all():
        raise ValueError(f'{path.name}: gap_under_005 no coincide con gap < 0.05')
    frames.append(frame.assign(source_csv=path.name))

results = pd.concat(frames, ignore_index=True)
for column in ['snapshot', 'raw_sha256', 'cv']:
    if results[column].isna().any() or results[column].nunique(dropna=False) != 1:
        raise ValueError(f'CSV no comparables: {column} debe ser el mismo y no nulo en todas las filas')
results = results.sort_values('cv_validation_f1_macro', ascending=False, kind='stable').reset_index(drop=True)
print(f'{len(paths)} archivos de comparación, {len(results)} filas. Orden por F1 macro medio de validación.')
print('Sin resultados del test reservado.')
display(results)


4 archivos de comparación, 7 filas. Orden por F1 macro medio de validación.
Sin resultados del test reservado.


,model,cv_train_f1_macro,cv_validation_f1_macro,cv_validation_std,gap,gap_under_005,cv_fit_seconds,evaluation_set,oof_accuracy,oof_f1_macro,oof_extreme_error_rate,oof_seconds,search_seconds,params,cv,snapshot,code_commit,raw_sha256,source_csv
0,XGBoost,0.811355,0.763119,0.003241,4.823581e-02,True,3.324304,train_oof,0.762986,0.763147,0.010111,3.397668,142.377483,"{""colsample_bytree"": 0.6645760092625056, ""eval...","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,6cf7cb7ff52d28460b08fda78144af39c70fad99,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,xgboost_comparison.csv
1,LightGBM,0.812453,0.762985,0.001777,4.946740e-02,True,1.844113,train_oof,0.762787,0.763016,0.009715,2.041968,222.518716,"{""boosting_type"": ""gbdt"", ""class_weight"": null...","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,8feaee5f5cc7702979ae9b073e382833bf26e521,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,lightgbm_comparison.csv
2,RandomForest,0.805477,0.759354,0.005580,4.612349e-02,True,8.610717,train_oof,0.758822,0.759376,0.009715,8.781747,1026.253963,"{""bootstrap"": true, ""max_depth"": 11, ""max_feat...","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,9bb15b1ffc5832592dcf023343150348ff6288e9,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,rf_comparison.csv
3,LogisticRegression,0.739585,0.729123,0.006091,1.046206e-02,True,0.668796,train_oof,0.729679,0.729183,0.012490,0.650682,20.194710,"{""C"": 0.5971896122019431, ""class_weight"": null}","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,5be5a63fcec6e779b070efb7d08027f5534c9bb3,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,logreg_comparison.csv
4,DummyClassifier,0.166791,0.166791,0.000068,8.618269e-09,True,0.236735,train_oof,0.333664,0.166791,0.333366,0.296802,0.000000,"{""strategy"": ""most_frequent""}","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,5be5a63fcec6e779b070efb7d08027f5534c9bb3,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,logreg_comparison.csv
5,DummyClassifier,0.166791,0.166791,0.000068,8.618269e-09,True,0.238719,train_oof,0.333664,0.166791,0.333366,0.306254,0.000000,"{""strategy"": ""most_frequent""}","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,9bb15b1ffc5832592dcf023343150348ff6288e9,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,rf_comparison.csv
6,DummyClassifier,0.166791,0.166791,0.000068,8.618269e-09,True,0.199005,train_oof,0.333664,0.166791,0.333366,0.222261,0.000000,"{""strategy"": ""most_frequent""}","StratifiedKFold(5, shuffle=True, random_state=42)",2026-09-17,6cf7cb7ff52d28460b08fda78144af39c70fad99,226f5e6af36cc82286765cda452cbe9abd6fbd70415aaf...,xgboost_comparison.csv


## Tabla para la presentación
Una fila por modelo (la referencia Dummy aparece en varios CSV con el mismo resultado; se muestra una vez). Todo es validación cruzada en train: el test no interviene.

In [2]:
presentation = (
    results.drop_duplicates('model')
    .assign(
        f1_val=lambda d: d['cv_validation_f1_macro'].map('{:.3f}'.format)
        + ' ± ' + d['cv_validation_std'].map('{:.3f}'.format),
        extreme=lambda d: (100 * d['oof_extreme_error_rate']).map('{:.1f} %'.format),
        gap_ok=lambda d: d['gap_under_005'].map({True: 'sí', False: 'no', 'True': 'sí', 'False': 'no'}),
    )
    [['model', 'cv_train_f1_macro', 'f1_val', 'gap', 'gap_ok', 'extreme', 'cv_fit_seconds', 'search_seconds']]
    .rename(columns={
        'model': 'Modelo',
        'cv_train_f1_macro': 'F1 macro train',
        'f1_val': 'F1 macro validación (media ± std)',
        'gap': 'Gap train − validación',
        'gap_ok': 'Gap < 0,05',
        'extreme': 'Errores extremos (OOF)',
        'cv_fit_seconds': 'Ajuste CV 5 folds (s)',
        'search_seconds': 'Búsqueda Optuna (s)',
    })
    .round({'F1 macro train': 3, 'Gap train − validación': 3, 'Ajuste CV 5 folds (s)': 1, 'Búsqueda Optuna (s)': 0})
    .reset_index(drop=True)
)
presentation.index = presentation.index + 1
display(presentation)

,Modelo,F1 macro train,F1 macro validación (media ± std),Gap train − validación,"Gap < 0,05",Errores extremos (OOF),Ajuste CV 5 folds (s),Búsqueda Optuna (s)
1,XGBoost,0.811,0.763 ± 0.003,0.048,sí,1.0 %,3.3,142.0
2,LightGBM,0.812,0.763 ± 0.002,0.049,sí,1.0 %,1.8,223.0
3,RandomForest,0.805,0.759 ± 0.006,0.046,sí,1.0 %,8.6,1026.0
4,LogisticRegression,0.740,0.729 ± 0.006,0.010,sí,1.2 %,0.7,20.0
5,DummyClassifier,0.167,0.167 ± 0.000,0.000,sí,33.3 %,0.2,0.0


## Elección del champion
Regla acordada antes de ver los resultados: **mejor F1 macro medio de validación entre los modelos con gap < 0,05** (sin contar la referencia Dummy). Se aplica tal cual; si la diferencia con el segundo es menor que la variación entre folds, se declara como empate práctico, pero no se cambia la regla después de ver los resultados.

In [3]:
GAP_LIMIT = 0.05
candidates = results[(results['model'] != 'DummyClassifier') & (results['gap'] < GAP_LIMIT)]
if candidates.empty:
    raise ValueError('Ningún modelo cumple gap < 0,05: no hay champion con la regla acordada')
champion = candidates.iloc[0]  # results ya está ordenado por F1 macro medio de validación
print(f"Champion: {champion['model']} · F1 macro validación {champion['cv_validation_f1_macro']:.4f} "
      f"± {champion['cv_validation_std']:.4f} · gap {champion['gap']:.4f}")

if len(candidates) > 1:
    runner_up = candidates.iloc[1]
    difference = champion['cv_validation_f1_macro'] - runner_up['cv_validation_f1_macro']
    noise = max(champion['cv_validation_std'], runner_up['cv_validation_std'])
    print(f"Segundo: {runner_up['model']} · {runner_up['cv_validation_f1_macro']:.4f} "
          f"± {runner_up['cv_validation_std']:.4f} · diferencia {difference:.4f}")
    if difference < noise:
        print(f'La diferencia es menor que la desviación entre folds ({noise:.4f}): empate práctico. '
              'Se mantiene la regla acordada.')

Champion: XGBoost · F1 macro validación 0.7631 ± 0.0032 · gap 0.0482
Segundo: LightGBM · 0.7630 ± 0.0018 · diferencia 0.0001
La diferencia es menor que la desviación entre folds (0.0032): empate práctico. Se mantiene la regla acordada.


## Conclusión
- **Champion: XGBoost** (F1 macro de validación 0,7631 ± 0,0032, gap 0,048), por la regla acordada; confirmado por el equipo el 17/11/2026.
- **LightGBM** queda prácticamente empatado (0,7630 ± 0,0018, gap 0,0495): la diferencia (0,0001) es mucho menor que la variación entre folds. En los mismos cinco folds, XGBoost gana en dos, LightGBM en dos y uno queda igualado (ver `xgboost_details.json` y `lightgbm_details.json`). Cualquiera de los dos sería defendible; se mantiene la regla fijada antes de ver los resultados.
- **Random Forest** (0,759) queda algo por detrás, con más variación entre folds y el ajuste más lento.
- **Regresión logística** (0,729) es la referencia lineal: peor F1 pero el gap más bajo (0,010).
- Todos superan claramente a la referencia Dummy (0,167).
- Son resultados de validación cruzada en train. La evaluación final del champion en test está en `08b_champion_test.ipynb`.